# Notebook 01 — Setup & Field Semantics

**Goal**: Connect to WorldQuant Brain, fetch all data fields for each dataset, 
generate LLM-powered semantic descriptions, and store them in the Chroma vector store.

Run this notebook **once** before anything else. Results are persisted to disk.

## Prerequisites
```bash
pip install -e ".[dev]"
cp .env.example .env   # fill in WQB_USERNAME, WQB_PASSWORD, LLM_MODEL, OPENAI_API_KEY
```

In [8]:
import asyncio
import nest_asyncio

nest_asyncio.apply()  # allow asyncio in Jupyter

from alpha_agent.config import settings

settings.ensure_dirs()

print("LLM model:", settings.llm_model)
print("Chroma dir:", settings.chroma_persist_dir)
print("DuckDB path:", settings.duckdb_path)
print("WQB username:", settings.wqb_username or "⚠ NOT SET")

LLM model: deepseek/deepseek-reasoner
Chroma dir: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/chroma
DuckDB path: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/alpha_memory.db
WQB username: freedom8625@163.com


## 1. Initialize the vector store

In [9]:
from alpha_agent.knowledge.vector_store import VectorStore

store = VectorStore()
print("Vector store initialized at:", settings.chroma_persist_dir)

Vector store initialized at: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/chroma


## 2. Connect to WQB and fetch data fields

In [10]:
import hashlib
import json
from pathlib import Path

from alpha_agent.data.wqb_client import WQBClient
from alpha_agent.data.datafield_indexer import DataFieldIndexer
from alpha_agent.knowledge.vector_store import COLLECTION_FIELDS

DATASETS = [
    {"id": "fundamental6", "universe": "TOP3000"},
    {"id": "analyst4",     "universe": "TOP1000"},
    {"id": "pv1",          "universe": "TOP1000"},
]

# 是否强制重新从 WQB 抓取并覆盖本地缓存
FORCE_REFRESH = False
# 命中本地缓存时，是否回灌到 Chroma（适合重建向量库后使用）
REINDEX_CACHE_TO_CHROMA = False

CACHE_PATH = settings.duckdb_path.parent / "dataset_fields_cache.json"

def _load_cache(cache_path: Path) -> dict:
    if not cache_path.exists():
        return {}
    try:
        return json.loads(cache_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[cache] Failed to read cache, will refetch: {e}")
        return {}

def _save_cache(cache_path: Path, payload: dict) -> None:
    cache_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

def _cache_key(dataset_id: str, universe: str) -> str:
    return f"{dataset_id}::{universe}"

def _upsert_cached_to_chroma(dataset_id: str, universe: str, cached_fields: list[dict]) -> None:
    ids, docs, metas = [], [], []
    for f in cached_fields:
        field_id = f.get("id", "")
        if not field_id:
            continue
        doc_id = f.get("doc_id") or hashlib.md5(f"{dataset_id}:{field_id}".encode()).hexdigest()
        document = f.get("document") or (
            f"Field: {field_id}\nDataset: {dataset_id}\n"
            f"Description: {f.get('description', '')}\n"
            f"Semantics: {f.get('semantics_json', '{}')}"
        )
        ids.append(doc_id)
        docs.append(document)
        metas.append({
            "field_id": field_id,
            "dataset": dataset_id,
            "universe": universe,
            "field_type": f.get("type", ""),
        })

    if ids:
        store.upsert(COLLECTION_FIELDS, ids=ids, documents=docs, metadatas=metas)

async def index_all_datasets(
    force_refresh: bool = False,
    reindex_cache_to_chroma: bool = False,
):
    cache = _load_cache(CACHE_PATH)
    all_enriched = {}
    need_fetch = []

    for ds in DATASETS:
        ds_id = ds["id"]
        uni = ds["universe"]
        key = _cache_key(ds_id, uni)
        if not force_refresh and key in cache:
            cached = cache[key]
            all_enriched[ds_id] = cached
            print(f"\n=== Cache hit: {ds_id} ({uni}) ===")
            print(f"  Loaded {len(cached)} fields from local cache")
            if reindex_cache_to_chroma:
                _upsert_cached_to_chroma(ds_id, uni, cached)
                print("  Reindexed cached fields into Chroma")
        else:
            need_fetch.append(ds)

    if need_fetch:
        async with WQBClient() as client:
            indexer = DataFieldIndexer(client, store)
            for ds in need_fetch:
                ds_id = ds["id"]
                uni = ds["universe"]
                print(f"\n=== Indexing dataset: {ds_id} ({uni}) ===")
                enriched = await indexer.index_dataset(
                    dataset_id=ds_id,
                    universe=uni,
                    force_refresh=force_refresh,
                )
                all_enriched[ds_id] = enriched
                cache[_cache_key(ds_id, uni)] = enriched
                print(f"  Indexed {len(enriched)} fields")

        _save_cache(CACHE_PATH, cache)
        print(f"\n[cache] Saved cache to: {CACHE_PATH}")
    else:
        print(f"\n[cache] All datasets loaded from cache: {CACHE_PATH}")

    return all_enriched

all_fields = asyncio.run(
    index_all_datasets(
        force_refresh=FORCE_REFRESH,
        reindex_cache_to_chroma=REINDEX_CACHE_TO_CHROMA,
    )
)


=== Cache hit: fundamental6 (TOP3000) ===
  Loaded 48 fields from local cache

=== Cache hit: analyst4 (TOP1000) ===
  Loaded 1 fields from local cache

=== Cache hit: pv1 (TOP1000) ===
  Loaded 12 fields from local cache

[cache] All datasets loaded from cache: /Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/data/dataset_fields_cache.json


## 3. Explore the indexed field semantics

In [11]:
# Count fields indexed per dataset
from alpha_agent.knowledge.vector_store import COLLECTION_FIELDS

total = store.count(COLLECTION_FIELDS)
print(f"Total fields indexed in vector store: {total}")

for ds_id, fields in all_fields.items():
    print(f"  {ds_id}: {len(fields)} fields")

Total fields indexed in vector store: 362
  fundamental6: 48 fields
  analyst4: 1 fields
  pv1: 12 fields


In [12]:
# Semantic search test
query = "earnings quality and profitability"
results = store.query(COLLECTION_FIELDS, query, k=5)

print(f"Top 5 fields for query: '{query}'\n")
for r in results:
    print(f"  [{r['metadata'].get('field_id')}] dist={r['distance']:.3f}")
    print(f"  {r['document'][:150]}\n")

Top 5 fields for query: 'earnings quality and profitability'

  [eps] dist=0.507
  Field: eps
Dataset: fundamental6
Description: Earnings Per Share (Basic) - Including Extraordinary Items
Semantics: {
  "category": "Fundamental",
  "

  [actual_eps_value_quarterly] dist=0.547
  actual_eps_value_quarterly

  [fnd6_cptmfmq_opepsq] dist=0.550
  Field: fnd6_cptmfmq_opepsq
Dataset: fundamental6
Description: Earnings Per Share from Operations
Semantics: {
  "category": "Fundamental",
  "meaning"

  [fnd6_cptnewqv1300_opepsq] dist=0.551
  Field: fnd6_cptnewqv1300_opepsq
Dataset: fundamental6
Description: Earnings Per Share from Operations
Semantics: {
  "category": "Fundamental",
  "mea

  [fnd6_spce] dist=0.551
  Field: fnd6_spce
Dataset: fundamental6
Description: S&P Core Earnings
Semantics: ```json
{
  "category": "Fundamental",
  "meaning": "Standardized ear



In [13]:
# Try another semantic query
query2 = "short-term price momentum and volume"
results2 = store.query(COLLECTION_FIELDS, query2, k=5)

print(f"Top 5 fields for query: '{query2}'\n")
for r in results2:
    print(f"  [{r['metadata'].get('field_id')}] dist={r['distance']:.3f}")
    print(f"  {r['document'][:150]}\n")

Top 5 fields for query: 'short-term price momentum and volume'

  [volume] dist=0.576
  volume

  [equity] dist=0.663
  equity

  [open] dist=0.706
  Field: open
Dataset: pv1
Description: Daily open price
Semantics: {
  "category": "Price/Volume",
  "meaning": "The price at which a security first tr

  [returns] dist=0.713
  Field: returns
Dataset: pv1
Description: Daily returns
Semantics: ```json
{
    "category": "Price/Volume",
    "meaning": "The daily percentage chang

  [fnd6_prchq] dist=0.721
  Field: fnd6_prchq
Dataset: fundamental6
Description: Price High - Quarter
Semantics: ```json
{
    "category": "Fundamental",
    "meaning": "The high



## 4. Initialize AlphaMemory (DuckDB)

Creates the database and tables. Will be empty on first run.

In [14]:
from alpha_agent.knowledge.alpha_memory import AlphaMemory

memory = AlphaMemory()
stats = memory.stats()
print("Alpha memory initialized")
print(f"  Total alphas: {stats['total']}")
print(f"  Qualified:    {stats['qualified']}")
print(f"  Pass rate:    {stats['pass_rate']}")

Alpha memory initialized
  Total alphas: 0
  Qualified:    0
  Pass rate:    0.0


In [ ]:
# Optional: release DB/file handles before switching notebooks
import gc

for obj_name in ["memory", "registry", "store", "pi"]:
    obj = globals().get(obj_name)
    close_fn = getattr(obj, "close", None)
    if callable(close_fn):
        try:
            close_fn()
            print(f"Closed: {obj_name}")
        except Exception as e:
            print(f"Failed to close {obj_name}: {e}")

for obj_name in ["memory", "registry", "store", "pi"]:
    if obj_name in globals():
        del globals()[obj_name]

gc.collect()
print("Released DB/file handles and cleared related globals.")

## ✅ Notebook 01 Complete

You now have:
- Data fields indexed in Chroma with LLM-generated semantics
- AlphaMemory (DuckDB) initialized and ready

**Next**: Run `02_build_operator_and_paper_kb.ipynb` to index operators and research papers.